# Shift-wise Pipeline — Self-contained Kaggle Notebook
All helper code is inlined here; no separate dataset / code upload required.
Just attach the competition dataset and run all cells.

In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/competitions/mindshift-analytics-haul-mark-challenge')
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(parents=True, exist_ok=True)

ENABLE_SPATIAL = False   # set True to try gpkg zone + hotspot features
REFUEL_SHIFT_MINUTES = 2
BASELINE_OOF_RMSE = 46.26
FOLDS = 5
SEED = 42

print('OUT_DIR:', OUT_DIR)
print('ENABLE_SPATIAL:', ENABLE_SPATIAL)

## Helper functions (inlined from kaggle_make_submission.py)

In [ ]:
from __future__ import annotations

import os
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------
# Train / test window splits
# --------------------------

TRAIN_WINDOWS = [
    (pd.Timestamp('2026-01-01'), pd.Timestamp('2026-01-20')),
    (pd.Timestamp('2026-02-01'), pd.Timestamp('2026-02-20')),
    (pd.Timestamp('2026-03-01'), pd.Timestamp('2026-03-11')),
]
TEST_WINDOWS = [
    (pd.Timestamp('2026-01-21'), pd.Timestamp('2026-01-31')),
    (pd.Timestamp('2026-02-21'), pd.Timestamp('2026-02-28')),
    (pd.Timestamp('2026-03-12'), pd.Timestamp('2026-03-20')),
]


def _parse_file_window(path: Path):
    m = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{4}-\d{2}-\d{2})', path.name)
    if not m:
        return None
    return pd.Timestamp(m.group(1)), pd.Timestamp(m.group(2))


def _in_any_window(start, end, windows):
    for w0, w1 in windows:
        if start >= w0 and end <= w1:
            return True
    return False


def split_train_test_files(files):
    train_files, test_files = [], []
    for f in files:
        wnd = _parse_file_window(f)
        if wnd is None:
            continue
        start, end = wnd
        if _in_any_window(start, end, TRAIN_WINDOWS):
            train_files.append(f)
        elif _in_any_window(start, end, TEST_WINDOWS):
            test_files.append(f)
    return sorted(train_files), sorted(test_files)


def discover_files(data_dir: Path):
    telemetry = sorted([p for p in data_dir.rglob('telemetry_*.parquet') if p.is_file()])
    if not telemetry:
        telemetry = sorted([p for p in data_dir.rglob('telemetry_*.csv') if p.is_file()])
    refuels = sorted(data_dir.rglob('rfid_refuels_*.parquet'))
    smry = sorted(data_dir.rglob('smry_*_train_ordered.csv'))
    id_map_new = next(iter(sorted(data_dir.rglob('id_mapping_new.csv'))), None)
    id_map = next(iter(sorted(data_dir.rglob('id_mapping.csv'))), None)
    gpkg = sorted(data_dir.rglob('*.gpkg'))
    return {
        'telemetry': telemetry,
        'refuels': refuels,
        'summary': smry,
        'id_mapping_new': id_map_new,
        'id_mapping': id_map,
        'gpkg': gpkg,
    }


def resolve_data_dir(candidate: Path) -> Path:
    if not candidate.exists() or not candidate.is_dir():
        return candidate
    nested = candidate / 'data'
    if nested.is_dir():
        nested_files = discover_files(nested)
        if nested_files['telemetry'] or nested_files['summary'] or nested_files['refuels']:
            return nested
    return candidate


# --------------------------
# Time helpers
# --------------------------

def ensure_local_ts(ts: pd.Series) -> pd.Series:
    out = pd.to_datetime(ts, errors='coerce')
    if out.dt.tz is None:
        out = out.dt.tz_localize('UTC')
    return out.dt.tz_convert('Asia/Kolkata')


def add_shift_and_workdate(df: pd.DataFrame, ts_col: str = 'ts') -> pd.DataFrame:
    out = df.copy()
    hour = out[ts_col].dt.hour
    out['shift'] = np.where((hour >= 6) & (hour < 14), 'A',
                   np.where((hour >= 14) & (hour < 22), 'B', 'C'))
    day0 = out[ts_col].dt.tz_localize(None).dt.normalize()
    is_night_carry = (out['shift'] == 'C') & (hour < 6)
    work_day = day0 - pd.to_timedelta(is_night_carry.astype('int8'), unit='D')
    out['date'] = work_day.dt.date
    return out


# --------------------------
# Spatial (optional)
# --------------------------

def mine_gpkg_map(gpkg_files):
    out = {}
    for g in gpkg_files:
        name = g.name.lower()
        if 'mine_001' in name or 'mine001' in name:
            out['mine001'] = g
        elif 'mine_002' in name or 'mine002' in name:
            out['mine002'] = g
    return out


# --------------------------
# Refuel features
# --------------------------

def load_refuels(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path)


def aggregate_refuels_by_shift(refuels: pd.DataFrame, minutes_shift: int) -> pd.DataFrame:
    r = refuels.copy()
    if 'fleet_type' in r.columns:
        r = r[r['fleet_type'] == 'Dumper'].copy()

    r['ts_local'] = ensure_local_ts(r['ts'])
    r['ts_local_shifted'] = r['ts_local'] + pd.Timedelta(minutes=minutes_shift)

    hour = r['ts_local_shifted'].dt.hour
    r['shift'] = np.where((hour >= 6) & (hour < 14), 'A',
                 np.where((hour >= 14) & (hour < 22), 'B', 'C'))
    day0 = r['ts_local_shifted'].dt.tz_localize(None).dt.normalize()
    is_night_carry = (r['shift'] == 'C') & (hour < 6)
    work_day = day0 - pd.to_timedelta(is_night_carry.astype('int8'), unit='D')
    r['date'] = work_day.dt.date

    r = r.sort_values(['vehicle', 'ts_local']).copy()
    r['prev_refuel_ts_local'] = r.groupby('vehicle', observed=True)['ts_local'].shift(1)
    r['hrs_since_prev_refuel'] = (r['ts_local'] - r['prev_refuel_ts_local']).dt.total_seconds() / 3600.0

    agg = (
        r.groupby(['vehicle', 'date', 'shift'], observed=True)
        .agg(
            refuel_liters=('litres', 'sum'),
            refuel_count=('litres', 'count'),
            refuel_liters_max=('litres', 'max'),
            hrs_since_prev_refuel_min=('hrs_since_prev_refuel', 'min'),
            hrs_since_prev_refuel_mean=('hrs_since_prev_refuel', 'mean'),
        )
        .reset_index()
        .fillna(0)
    )

    suffix = f'shift_{minutes_shift:+d}m'.replace('+', 'p').replace('-', 'm')
    return agg.rename(columns={
        'refuel_liters': f'refuel_liters_{suffix}',
        'refuel_count': f'refuel_count_{suffix}',
        'refuel_liters_max': f'refuel_liters_max_{suffix}',
        'hrs_since_prev_refuel_min': f'hrs_since_prev_refuel_min_{suffix}',
        'hrs_since_prev_refuel_mean': f'hrs_since_prev_refuel_mean_{suffix}',
    })


# --------------------------
# Telemetry -> shift features
# --------------------------

def build_shift_features_from_telemetry(
    telemetry_paths,
    *,
    enable_spatial: bool,
    gpkg_by_mine: dict,
    vehicles_keep=None,
) -> pd.DataFrame:
    out_chunks = []

    spatial_ctx = None
    if enable_spatial:
        try:
            import geopandas as gpd
            import numpy as _np
            from pyproj import Transformer

            transformer = Transformer.from_crs('EPSG:4326', 'EPSG:32645', always_xy=True)

            def load_layers(gpkg_path):
                layers = {}
                for lname in gpd.list_layers(gpkg_path).name.tolist():
                    gdf = gpd.read_file(gpkg_path, layer=lname)
                    if gdf.crs is None:
                        gdf = gdf.set_crs('EPSG:32645')
                    elif str(gdf.crs).upper() != 'EPSG:32645':
                        gdf = gdf.to_crs('EPSG:32645')
                    layers[lname] = gdf
                return layers

            def safe_union(gdf, buffer_m: float):
                if gdf is None or getattr(gdf, 'empty', True):
                    return None
                return gdf.geometry.buffer(buffer_m).union_all()

            mine_layers = {m: load_layers(p) for m, p in gpkg_by_mine.items()}
            spatial_ctx = {
                'gpd': gpd, 'np': _np, 'transformer': transformer,
                'mine_layers': mine_layers, 'safe_union': safe_union,
            }
        except Exception as e:
            print(f'[spatial] disabled due to import/load error: {e}')
            spatial_ctx = None

    for path in telemetry_paths:
        print(f'[telemetry] reading {path.name}')

        use_cols = [
            'vehicle', 'mine_anon', 'ts', 'received_ts',
            'latitude', 'longitude', 'altitude', 'speed', 'ignition',
            'disthav', 'analog_input_1', 'satellites', 'gnss_hdop',
            'angle', 'battery_level', 'axis_x', 'axis_y', 'axis_z',
        ]
        df = pd.read_parquet(path, columns=use_cols)
        df = df[[c for c in use_cols if c in df.columns]].copy()

        if vehicles_keep is not None and 'vehicle' in df.columns:
            df = df[df['vehicle'].astype(str).isin(vehicles_keep)]
            if df.empty:
                continue

        optional_defaults = {
            'mine_anon': None, 'received_ts': None,
            'latitude': np.nan, 'longitude': np.nan,
            'analog_input_1': 0.0, 'satellites': np.nan,
            'gnss_hdop': np.nan, 'angle': np.nan,
            'battery_level': np.nan,
            'axis_x': 0.0, 'axis_y': 0.0, 'axis_z': 0.0,
        }
        for c, default in optional_defaults.items():
            if c not in df.columns:
                df[c] = default

        mandatory = {'vehicle', 'ts', 'speed', 'disthav', 'ignition', 'altitude'}
        if not mandatory.issubset(set(df.columns)):
            print(f'[telemetry] skipping {path.name} (missing {sorted(mandatory - set(df.columns))})')
            continue

        for c in ['speed', 'altitude', 'disthav', 'analog_input_1', 'gnss_hdop',
                  'angle', 'battery_level', 'axis_x', 'axis_y', 'axis_z']:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')
        for c in ['ignition', 'satellites']:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype('int16')
        if 'vehicle' in df.columns:
            df['vehicle'] = df['vehicle'].astype(str)

        df['ts'] = ensure_local_ts(df['ts'])
        if 'received_ts' in df.columns:
            df['received_ts'] = ensure_local_ts(df['received_ts'])
            df['rx_delay_s'] = (df['received_ts'] - df['ts']).dt.total_seconds().clip(lower=0, upper=3600)
        else:
            df['rx_delay_s'] = 0.0

        df = add_shift_and_workdate(df, 'ts')
        df = df[df['speed'] <= 80].copy()
        df['speed'] = df['speed'].clip(lower=0, upper=60)
        df = df.sort_values(['vehicle', 'ts'], kind='mergesort').copy()

        gkey = ['vehicle', 'date', 'shift']
        df['time_gap_s'] = (df.groupby(gkey, observed=True, sort=False)['ts']
                             .diff().dt.total_seconds().fillna(0).clip(0, 300))
        df['alt_diff'] = df.groupby(gkey, observed=True)['altitude'].diff().fillna(0)
        df['alt_climb'] = df['alt_diff'].clip(lower=0)
        df['alt_descent'] = (-df['alt_diff']).clip(lower=0)

        prev_speed = df.groupby(gkey, observed=True, sort=False)['speed'].shift(1).fillna(0)
        df['stop_event'] = ((prev_speed > 2) & (df['speed'] <= 2) & (df['ignition'] == 1)).astype('int8')

        prev_a = df.groupby(gkey, observed=True, sort=False)['analog_input_1'].shift(1)
        df['dump_edge'] = ((df['analog_input_1'] > 2.5) & ((prev_a <= 2.5) | prev_a.isna())).astype('int8')
        df['dump_high'] = (df['analog_input_1'] > 2.5).astype('int8')

        df['ignition_on_time_s'] = df['time_gap_s'] * (df['ignition'] == 1)
        df['moving_time_s'] = df['time_gap_s'] * (df['speed'] > 2)
        df['idle_time_s'] = df['time_gap_s'] * ((df['ignition'] == 1) & (df['speed'] <= 2))
        df['dump_high_time_s'] = df['time_gap_s'] * (df['dump_high'] == 1)

        # Spatial zone flags (optional)
        if (spatial_ctx is not None
                and 'latitude' in df.columns
                and 'longitude' in df.columns
                and 'mine_anon' in df.columns):
            try:
                gpd = spatial_ctx['gpd']
                _np = spatial_ctx['np']
                transformer = spatial_ctx['transformer']
                mine_layers = spatial_ctx['mine_layers']
                safe_union = spatial_ctx['safe_union']

                x, y = transformer.transform(df['longitude'].to_numpy(), df['latitude'].to_numpy())
                df['x_utm'] = x
                df['y_utm'] = y

                for c in ['in_dump_zone', 'in_ob_dump_zone', 'in_rom_stock_zone', 'in_load_zone', 'on_haul_road']:
                    df[c] = 0

                parts = []
                for mine, part in df.groupby('mine_anon', observed=True):
                    mine_key = str(mine).lower()
                    layers = mine_layers.get(mine_key)
                    if layers is None:
                        parts.append(part)
                        continue

                    points = gpd.GeoSeries(
                        gpd.points_from_xy(part['x_utm'], part['y_utm']), crs='EPSG:32645'
                    )
                    ob_dump_union = safe_union(layers.get('ob_dump'), 20.0)
                    stock_union = safe_union(layers.get('stock', layers.get('mineral_stock')), 20.0)
                    bench_union = safe_union(layers.get('bench'), 20.0)
                    cpu_union = safe_union(layers.get('cpu'), 20.0)
                    haul_union = safe_union(layers.get('haul_road'), 15.0)

                    in_ob = points.within(ob_dump_union).to_numpy() if ob_dump_union is not None else _np.zeros(len(part), dtype=bool)
                    in_rom = points.within(stock_union).to_numpy() if stock_union is not None else _np.zeros(len(part), dtype=bool)
                    in_dump = in_ob | in_rom

                    in_load = _np.zeros(len(part), dtype=bool)
                    for zone in [bench_union, cpu_union]:
                        if zone is not None:
                            in_load |= points.within(zone).to_numpy()

                    on_haul = points.within(haul_union).to_numpy() if haul_union is not None else _np.zeros(len(part), dtype=bool)

                    part = part.copy()
                    part['in_dump_zone'] = in_dump.astype('int8')
                    part['in_ob_dump_zone'] = in_ob.astype('int8')
                    part['in_rom_stock_zone'] = in_rom.astype('int8')
                    part['in_load_zone'] = in_load.astype('int8')
                    part['on_haul_road'] = on_haul.astype('int8')
                    parts.append(part)

                df = pd.concat(parts, ignore_index=True)
            except Exception as e:
                print(f'[spatial] failed on {path.name}: {e}')

        # Aggregations
        agg = df.groupby(gkey, observed=True).agg(
            n_pings=('ts', 'size'),
            ignition_on_time_s=('ignition_on_time_s', 'sum'),
            moving_time_s=('moving_time_s', 'sum'),
            idle_time_s=('idle_time_s', 'sum'),
            dump_high_time_s=('dump_high_time_s', 'sum'),
            disthav_m=('disthav', 'sum'),
            altitude_mean=('altitude', 'mean'),
            altitude_std=('altitude', 'std'),
            altitude_max=('altitude', 'max'),
            altitude_min=('altitude', 'min'),
            total_climb_m=('alt_climb', 'sum'),
            total_descent_m=('alt_descent', 'sum'),
            speed_mean=('speed', 'mean'),
            speed_std=('speed', 'std'),
            speed_max=('speed', 'max'),
            stop_count=('stop_event', 'sum'),
            dump_event_count=('dump_edge', 'sum'),
            dump_signal_mean=('analog_input_1', 'mean'),
            dump_signal_std=('analog_input_1', 'std'),
            dump_signal_max=('analog_input_1', 'max'),
            avg_satellites=('satellites', 'mean'),
            avg_hdop=('gnss_hdop', 'mean'),
            vibration_x_std=('axis_x', 'std'),
            vibration_y_std=('axis_y', 'std'),
            vibration_z_std=('axis_z', 'std'),
            battery_mean=('battery_level', 'mean'),
            angle_std=('angle', 'std'),
            hour_start=('ts', lambda s: int(s.iloc[0].hour) if len(s) else 0),
            hour_end=('ts', lambda s: int(s.iloc[-1].hour) if len(s) else 0),
            rx_delay_mean_s=('rx_delay_s', 'mean'),
        ).reset_index()

        agg['ignition_on_hours'] = agg['ignition_on_time_s'] / 3600.0
        agg['moving_hours'] = agg['moving_time_s'] / 3600.0
        agg['idle_hours'] = agg['idle_time_s'] / 3600.0
        agg['dump_high_time_h'] = agg['dump_high_time_s'] / 3600.0
        agg['shift_km'] = agg['disthav_m'] / 1000.0
        agg.drop(columns=['ignition_on_time_s', 'moving_time_s', 'idle_time_s',
                           'dump_high_time_s', 'disthav_m'], inplace=True)

        agg['altitude_range'] = agg['altitude_max'] - agg['altitude_min']
        agg['idle_fraction'] = agg['idle_hours'] / (agg['ignition_on_hours'] + 1e-6)
        agg['stop_density'] = agg['stop_count'] / (agg['shift_km'] + 1e-6)
        agg['km_per_hour'] = agg['shift_km'] / (agg['ignition_on_hours'] + 1e-6)
        agg['climb_descent_ratio'] = agg['total_climb_m'] / (agg['total_descent_m'] + 1.0)

        speed_pos = df.loc[df['speed'] > 0, gkey + ['speed']].copy()
        if len(speed_pos):
            q = speed_pos.groupby(gkey, observed=True)['speed'].quantile([0.5, 0.75, 0.9]).unstack(-1)
            q = q.rename(columns={0.5: 'speed_p50', 0.75: 'speed_p75', 0.9: 'speed_p90'}).reset_index()
            agg = agg.merge(q, on=gkey, how='left')
        else:
            agg['speed_p50'] = agg['speed_p75'] = agg['speed_p90'] = 0.0

        first_last = df.groupby(gkey, observed=True)['altitude'].agg(['first', 'last']).reset_index()
        first_last['net_lift'] = first_last['last'] - first_last['first']
        agg = agg.merge(first_last[gkey + ['net_lift']], on=gkey, how='left')

        agg[['vibration_x_std', 'vibration_y_std', 'vibration_z_std']] = agg[
            ['vibration_x_std', 'vibration_y_std', 'vibration_z_std']
        ].fillna(0)
        agg['vibration_magnitude'] = np.sqrt(
            agg['vibration_x_std'] ** 2 + agg['vibration_y_std'] ** 2 + agg['vibration_z_std'] ** 2
        )
        agg.drop(columns=['vibration_x_std', 'vibration_y_std', 'vibration_z_std'], inplace=True)

        # Spatial aggregates
        if 'in_dump_zone' in df.columns:
            for zc in ['in_dump_zone', 'in_ob_dump_zone', 'in_rom_stock_zone', 'in_load_zone', 'on_haul_road']:
                if zc in df.columns:
                    df[zc] = df[zc].fillna(0).astype('int8')

            zone_means = df.groupby(gkey, observed=True)[
                ['in_dump_zone', 'in_ob_dump_zone', 'in_rom_stock_zone', 'in_load_zone', 'on_haul_road']
            ].mean().reset_index().rename(columns={
                'in_dump_zone': 'frac_in_dump_zone',
                'in_ob_dump_zone': 'frac_in_ob_dump_zone',
                'in_rom_stock_zone': 'frac_in_rom_stock_zone',
                'in_load_zone': 'frac_in_load_zone',
                'on_haul_road': 'frac_on_haul_road',
            })
            agg = agg.merge(zone_means, on=gkey, how='left')

            df['time_in_dump_s'] = df['time_gap_s'] * df['in_dump_zone']
            df['time_in_ob_s'] = df['time_gap_s'] * df['in_ob_dump_zone']
            df['time_in_rom_s'] = df['time_gap_s'] * df['in_rom_stock_zone']
            df['time_in_load_s'] = df['time_gap_s'] * df['in_load_zone']
            df['time_on_haul_s'] = df['time_gap_s'] * df['on_haul_road']

            zone_time = df.groupby(gkey, observed=True).agg(
                total_time_s=('time_gap_s', 'sum'),
                time_in_dump_s=('time_in_dump_s', 'sum'),
                time_in_ob_s=('time_in_ob_s', 'sum'),
                time_in_rom_s=('time_in_rom_s', 'sum'),
                time_in_load_s=('time_in_load_s', 'sum'),
                time_on_haul_s=('time_on_haul_s', 'sum'),
            ).reset_index()
            for c in ['time_in_dump_s', 'time_in_ob_s', 'time_in_rom_s', 'time_in_load_s', 'time_on_haul_s']:
                zone_time[c.replace('_s', '_h')] = zone_time[c] / 3600.0
                zone_time[c.replace('_s', '_frac')] = zone_time[c] / (zone_time['total_time_s'] + 1e-6)
            zone_time = zone_time.drop(columns=['total_time_s'])
            agg = agg.merge(zone_time, on=gkey, how='left')

            df['dump_edge_in_ob'] = df['dump_edge'] * df['in_ob_dump_zone']
            df['dump_edge_in_rom'] = df['dump_edge'] * df['in_rom_stock_zone']
            de = df.groupby(gkey, observed=True).agg(
                dump_events_in_ob_dump=('dump_edge_in_ob', 'sum'),
                dump_events_in_rom_stock=('dump_edge_in_rom', 'sum'),
            ).reset_index()
            de = de.merge(agg[gkey + ['dump_event_count']], on=gkey, how='left')
            de['dump_events_outside_zones'] = (
                de['dump_event_count'] - de['dump_events_in_ob_dump'] - de['dump_events_in_rom_stock']
            ).clip(lower=0)
            de = de.drop(columns=['dump_event_count'])
            agg = agg.merge(de, on=gkey, how='left')

        # Hotspot dwell
        if 'x_utm' in df.columns and 'y_utm' in df.columns:
            grid = 50.0
            gx = np.floor(df['x_utm'] / grid).astype('int64')
            gy = np.floor(df['y_utm'] / grid).astype('int64')
            df['cell_id'] = gx * 1_000_000 + gy

            cell_dwell = df.groupby(gkey + ['cell_id'], observed=True)['time_gap_s'].sum().reset_index()
            top = cell_dwell.groupby(gkey, observed=True)['time_gap_s'].max().reset_index().rename(
                columns={'time_gap_s': 'hotspot_top_cell_dwell_s'}
            )
            uniq = cell_dwell.groupby(gkey, observed=True)['cell_id'].nunique().reset_index().rename(
                columns={'cell_id': 'hotspot_unique_cells'}
            )
            total = df.groupby(gkey, observed=True)['time_gap_s'].sum().reset_index().rename(
                columns={'time_gap_s': 'total_time_s'}
            )
            agg = agg.merge(top, on=gkey, how='left').merge(uniq, on=gkey, how='left').merge(total, on=gkey, how='left')
            agg['hotspot_top_cell_dwell_frac'] = agg['hotspot_top_cell_dwell_s'] / (agg['total_time_s'] + 1e-6)
            agg.drop(columns=['hotspot_top_cell_dwell_s', 'total_time_s'], inplace=True)
        else:
            agg['hotspot_unique_cells'] = 0
            agg['hotspot_top_cell_dwell_frac'] = 0.0

        out_chunks.append(agg)

    if not out_chunks:
        return pd.DataFrame(columns=['vehicle', 'date', 'shift'])
    return pd.concat(out_chunks, ignore_index=True)


# --------------------------
# Modeling utilities
# --------------------------

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


def get_feature_columns(df: pd.DataFrame, exclude: set) -> list:
    feats = []
    for c in df.columns:
        if c in exclude:
            continue
        if df[c].dtype == 'object':
            continue
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            continue
        feats.append(c)
    return feats


print('Helper functions loaded.')

## Build shift-wise features

In [ ]:
suffix = 'spatial' if ENABLE_SPATIAL else 'nospatial'
train_cache = OUT_DIR / f'train_shift_features__{suffix}.parquet'
test_cache  = OUT_DIR / f'test_shift_features__{suffix}.parquet'

DATA_DIR = resolve_data_dir(DATASET_ROOT)
print('DATA_DIR:', DATA_DIR)

if train_cache.exists() and test_cache.exists():
    print('Loading cached features...')
    train_feat = pd.read_parquet(train_cache)
    test_feat  = pd.read_parquet(test_cache)
else:
    files = discover_files(DATA_DIR)

    if not files['telemetry']:
        raise FileNotFoundError(f'No telemetry files found under: {DATA_DIR}')
    if not files['summary']:
        raise FileNotFoundError('No summary smry_*_train_ordered.csv found.')

    summary = pd.concat([pd.read_csv(p) for p in files['summary']], ignore_index=True)
    summary['vehicle'] = summary['vehicle'].astype(str)
    summary['date'] = pd.to_datetime(summary['date'], errors='coerce').dt.date

    id_map_path = files['id_mapping_new'] or files['id_mapping']
    if id_map_path is None:
        raise FileNotFoundError('No id_mapping_new.csv or id_mapping.csv found.')
    id_mapping = pd.read_csv(id_map_path)
    id_mapping['vehicle'] = id_mapping['vehicle'].astype(str)
    id_mapping['date'] = pd.to_datetime(id_mapping['date'], errors='coerce').dt.date

    vehicles_keep = set(
        pd.concat([summary['vehicle'], id_mapping['vehicle']], ignore_index=True).unique()
    )
    print('vehicles_keep:', len(vehicles_keep))

    train_paths, test_paths = split_train_test_files(files['telemetry'])
    print('Train windows:', len(train_paths))
    print('Test windows: ', len(test_paths))

    gpkg_by_mine = mine_gpkg_map(files['gpkg']) if ENABLE_SPATIAL else {}

    print('Building shift-wise features (train)...')
    train_feat = build_shift_features_from_telemetry(
        train_paths, enable_spatial=ENABLE_SPATIAL,
        gpkg_by_mine=gpkg_by_mine, vehicles_keep=vehicles_keep,
    )

    print('Building shift-wise features (test)...')
    test_feat = build_shift_features_from_telemetry(
        test_paths, enable_spatial=ENABLE_SPATIAL,
        gpkg_by_mine=gpkg_by_mine, vehicles_keep=vehicles_keep,
    )

    # Attach target
    train_feat = train_feat.merge(
        summary[['vehicle', 'date', 'shift', 'acons']],
        on=['vehicle', 'date', 'shift'], how='left'
    )

    # Refuel features
    if files['refuels']:
        ref_path = sorted(files['refuels'])[-1]
        print('Loading refuels:', ref_path.name)
        refuels = load_refuels(ref_path)

        ref_aggs = [aggregate_refuels_by_shift(refuels, 0)]
        if REFUEL_SHIFT_MINUTES:
            ref_aggs.append(aggregate_refuels_by_shift(refuels,  REFUEL_SHIFT_MINUTES))
            ref_aggs.append(aggregate_refuels_by_shift(refuels, -REFUEL_SHIFT_MINUTES))

        for agg in ref_aggs:
            train_feat = train_feat.merge(agg, on=['vehicle', 'date', 'shift'], how='left')
            test_feat  = test_feat.merge(agg,  on=['vehicle', 'date', 'shift'], how='left')

        ref_cols = [c for c in train_feat.columns
                    if c.startswith('refuel_') or c.startswith('hrs_since_prev_refuel')]
        train_feat[ref_cols] = train_feat[ref_cols].fillna(0)
        test_feat[ref_cols]  = test_feat[ref_cols].fillna(0)
    else:
        print('No refuels parquet found; skipping refuel features.')

    # Temporal features
    for df in (train_feat, test_feat):
        dt = pd.to_datetime(df['date'], errors='coerce')
        df['day_of_week']  = dt.dt.dayofweek
        df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
        df['week_number']  = dt.dt.isocalendar().week.astype(int)
        df['day_of_month'] = dt.dt.day.astype(int)
        df['shift_enc']    = df['shift'].map({'C': 0, 'A': 1, 'B': 2}).astype(int)

    # Vehicle aggregates (train-only statistics to avoid leakage)
    veh_stats = (
        train_feat.groupby('vehicle', observed=True)
        .agg(
            veh_mean_km=('shift_km', 'mean'),
            veh_mean_ignition_h=('ignition_on_hours', 'mean'),
            veh_mean_dumps=('dump_event_count', 'mean'),
            veh_mean_idle_frac=('idle_fraction', 'mean'),
            veh_mean_speed=('speed_max', 'mean'),
            veh_std_km=('shift_km', 'std'),
        )
        .reset_index()
    )
    train_feat = train_feat.merge(veh_stats, on='vehicle', how='left')
    test_feat  = test_feat.merge(veh_stats,  on='vehicle', how='left')

    print('Caching features...')
    train_feat.to_parquet(train_cache, index=False)
    test_feat.to_parquet(test_cache,   index=False)

print('train_feat:', train_feat.shape)
print('test_feat :', test_feat.shape)
train_feat.head(3)

## Train LightGBM + generate submission

In [ ]:
import pkgutil, subprocess, sys
if pkgutil.find_loader('lightgbm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'])
if pkgutil.find_loader('sklearn') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'])

import lightgbm as lgb
from sklearn.model_selection import GroupKFold

train_model_df = train_feat.dropna(subset=['acons']).copy()
exclude = {'vehicle', 'date', 'shift', 'acons'}
features = get_feature_columns(train_model_df, exclude)

X      = train_model_df[features].fillna(0)
y      = train_model_df['acons'].to_numpy(dtype=float)
groups = train_model_df['vehicle'].astype('category').cat.codes.to_numpy()

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.03,
    'num_leaves': 63,
    'max_depth': -1,
    'min_data_in_leaf': 25,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'lambda_l1': 0.0,
    'lambda_l2': 0.3,
    'verbosity': -1,
    'seed': SEED,
}

gkf = GroupKFold(n_splits=FOLDS)
oof = np.zeros(len(train_model_df), dtype=float)
models = []
fold_rmses = []

print('\n[CV] GroupKFold(vehicle)')
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    dtr = lgb.Dataset(X_tr, label=y_tr)
    dva = lgb.Dataset(X_va, label=y_va, reference=dtr)

    model = lgb.train(
        params, dtr,
        valid_sets=[dva],
        num_boost_round=4000,
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(200)],
    )
    pred = np.clip(model.predict(X_va, num_iteration=model.best_iteration), 0, None)
    oof[va_idx] = pred

    fold_rmse = rmse(y_va, pred)
    fold_rmses.append(fold_rmse)
    print(f'  fold {fold}: rmse={fold_rmse:.4f}')
    models.append(model)

overall = rmse(y, np.clip(oof, 0, None))
delta   = BASELINE_OOF_RMSE - overall
print(f'\nOOF RMSE: {overall:.6f}')
print(f'Fold RMSE mean±std: {np.mean(fold_rmses):.6f} ± {np.std(fold_rmses):.6f}')
print(f'Baseline OOF RMSE: {BASELINE_OOF_RMSE:.6f}')
print(f'Delta (baseline - new): {delta:+.6f}  (positive = improvement)')

In [ ]:
# Predict test set (ensemble average across folds)
X_test    = test_feat[features].fillna(0)
test_pred = np.mean(
    [m.predict(X_test, num_iteration=m.best_iteration) for m in models], axis=0
)
test_pred = np.clip(test_pred, 0, None)

test_pred_df = test_feat[['vehicle', 'date', 'shift']].copy()
test_pred_df['Predicted'] = test_pred

# Load id_mapping for final join
files = discover_files(DATA_DIR)
id_map_path = files['id_mapping_new'] or files['id_mapping']
id_mapping = pd.read_csv(id_map_path)
id_mapping['vehicle'] = id_mapping['vehicle'].astype(str)
id_mapping['date']    = pd.to_datetime(id_mapping['date'], errors='coerce').dt.date

submission = id_mapping.merge(test_pred_df, on=['vehicle', 'date', 'shift'], how='left')
missing_before = int(submission['Predicted'].isna().sum())

global_mean = float(train_model_df['acons'].mean())
submission['Predicted'] = submission['Predicted'].fillna(global_mean).clip(lower=0)

out_path = OUT_DIR / 'submission.csv'
submission[['id', 'Predicted']].sort_values('id').to_csv(out_path, index=False)

print('Saved:', out_path)
print(f'Submission rows: {len(submission)} | missing before fallback: {missing_before}')
print('Prediction stats:')
print(submission['Predicted'].describe())

## Download
After the notebook finishes, download `/kaggle/working/submission.csv` from the right-side **Output** panel.